<a href="https://colab.research.google.com/github/Angelinat3/244107020076-mobile-course/blob/main/JS04/Assignment_JS04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = pd.read_csv('insurance.csv')

# Dataset Description
print("Dataset Info:")
df.info()
print("\nDataset Description:")
display(df.describe())

# Visualizing relationships between features and the target variable (charges)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.scatterplot(x='age', y='charges', hue='smoker', data=df, ax=axes[0])
axes[0].set_title('Age vs Charges')

sns.scatterplot(x='bmi', y='charges', hue='smoker', data=df, ax=axes[1])
axes[1].set_title('BMI vs Charges')

sns.boxplot(x='children', y='charges', data=df, ax=axes[2])
axes[2].set_title('Children vs Charges')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Encoding categorical variables (sex, smoker, region)
df_encoded = pd.get_dummies(df, drop_first=True)

# 1. Identify Variables
# Independent variables (X) and target variable (y)
X = df_encoded.drop('charges', axis=1)
y = df_encoded['charges']

# 2. Partition the dataset (80:20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data successfully encoded, split, and scaled. ✅")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 4 & 5. Construct and Train Multiple Linear Regression
mlr_model = LinearRegression()
mlr_model.fit(X_train_scaled, y_train)

# Predict on testing data
y_pred_mlr = mlr_model.predict(X_test_scaled)

# 6. Evaluate the MLR model
print("=== Multiple Linear Regression Evaluation ===")
print(f"R-squared: {r2_score(y_test, y_pred_mlr):.4f}")
print(f"MSE: {mean_squared_error(y_test, y_pred_mlr):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_mlr):.4f}")

In [ ]:
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV

# Scale the target variable for SVR
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

# 7. SVR Model with Hyperparameter Tuning
svr = SVR()
param_grid = {
    'kernel': ['linear', 'rbf'],
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto']
}

grid_search = GridSearchCV(estimator=svr, param_grid=param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train_scaled)

# Get the best model and predict
best_svr = grid_search.best_estimator_
y_pred_svr_scaled = best_svr.predict(X_test_scaled)

# Inverse transform to get actual values
y_pred_svr = scaler_y.inverse_transform(y_pred_svr_scaled.reshape(-1, 1)).ravel()

# Evaluate the SVR model
print("=== Support Vector Regression (SVR) Evaluation ===")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"R-squared: {r2_score(y_test, y_pred_svr):.4f}")
print(f"MSE: {mean_squared_error(y_test, y_pred_svr):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_svr):.4f}")